In [11]:
import pandas as pd

In [12]:
price_dataset = pd.read_csv("../data/raw/entso-e-prices/Poland.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [13]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Poland,POL,2015-01-01 00:00:00,2015-01-01 01:00:00,20.42,2015-01-01 00:00:00
1,Poland,POL,2015-01-01 01:00:00,2015-01-01 02:00:00,20.42,2015-01-01 01:00:00
2,Poland,POL,2015-01-01 02:00:00,2015-01-01 03:00:00,20.42,2015-01-01 02:00:00
3,Poland,POL,2015-01-01 03:00:00,2015-01-01 04:00:00,20.42,2015-01-01 03:00:00
4,Poland,POL,2015-01-01 04:00:00,2015-01-01 05:00:00,20.42,2015-01-01 04:00:00


In [14]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [15]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,20.42,2015-01-01 00:00:00
1,20.42,2015-01-01 01:00:00
2,20.42,2015-01-01 02:00:00
3,20.42,2015-01-01 03:00:00
4,20.42,2015-01-01 04:00:00


In [16]:
solar_dataset = pd.read_csv("../data/raw/PL/solar-raw.csv")

In [17]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [18]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [19]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [20]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [21]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [22]:
meteo_dataset = pd.read_csv("../data/raw/meteo-raw-poland.csv")

In [23]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,9.3,22.3,280,100,0.0,0.7
1,2022-01-01T01:00,9.3,21.2,280,100,0.0,0.1
2,2022-01-01T02:00,8.9,20.5,281,100,0.0,0.9
3,2022-01-01T03:00,8.8,19.3,284,100,0.0,1.2
4,2022-01-01T04:00,8.8,18.5,294,100,0.0,1.5


In [24]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [25]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [26]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [27]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,9.3,22.3,280,100,0.0,0.7,2022-01-01 00:00:00
1,9.3,21.2,280,100,0.0,0.1,2022-01-01 01:00:00
2,8.9,20.5,281,100,0.0,0.9,2022-01-01 02:00:00
3,8.8,19.3,284,100,0.0,1.2,2022-01-01 03:00:00
4,8.8,18.5,294,100,0.0,1.5,2022-01-01 04:00:00


In [28]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,49.37,2022-01-01 00:00:00,0.0,9.3,22.3,280,100,0.0,0.7
1,43.22,2022-01-01 01:00:00,0.0,9.3,21.2,280,100,0.0,0.1
2,45.46,2022-01-01 02:00:00,0.0,8.9,20.5,281,100,0.0,0.9
3,37.67,2022-01-01 03:00:00,0.0,8.8,19.3,284,100,0.0,1.2
4,39.70,2022-01-01 04:00:00,0.0,8.8,18.5,294,100,0.0,1.5


In [29]:
merged.to_csv("../data/processed/poland_merged.csv", index=False)